# COS30049 A2 - Stage B3 tried on the round-1 152M model: does adding weak-case examples help? (T4 x2)

`docs/TRAINING_PLAN.md` stage B3, first tried on round 1 while the round-2 pretraining runs, so the recipe is known
before the round-2 model exists. From round 1's chat model, two explanation stages run side by side, one per GPU,
with `configs/text_150m_explain_v2.json`:

- **A**: DS7 v2 + the glossary x4 (round 2's planned recipe);
- **B**: the same + the weak-case examples (`explorer/weak_cases.py`: other methods, reworded instructions of its
  own, shuffled facts, three or four reasons).

Then round 1's writer, A and B answer the same dev prompts (the weak-case dev slice and a fixed draw of DS7 v2's
validation split). `python -m explorer.cli writer-gate` (run at home on the output tables) decides; the test sets
are not touched here.

**Settings:** Accelerator **GPU T4 x2**, **Internet on**. **Inputs:** this account's datasets
`cos30049a2-text150m-base` and `cos30049a2-text150m-chat-v1` (round 1's chat model) and notebook
`cos30049a2-text150m-stage-data`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "3978c8c5e938112cfa324066076b1e1878ee7c73"   # the code this run uses; recorded in kaggle_run.json
SESSION_HOURS = 11.75  # Kaggle ends a GPU session at 12 h
EVAL_HOURS = 1.5       # three writers on two dev files, one queue per GPU
# round 1's chat model (runs/text-150m-chat/model.pt on the PC), uploaded as dataset cos30049a2-text150m-chat-v1
CHAT_V1_SHA256 = "81645aa07bff32e2dc2ee8e94ceafcfeb85a2c0efd8bf2b75d4d826aabc9dfdc"
PINS = ["tokenizers==0.21.1", "numpy==2.2.6"]  # environment.yml; torch is Kaggle's own (recorded)

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
RUNS = WORK / "runs"

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
base = check_base()
stage_manifest_path = find_input("stage_manifest.json")
for rel, digest in json.loads(stage_manifest_path.read_text(encoding="utf-8"))["files"].items():
    assert sha256(stage_manifest_path.parent / rel) == digest, f"{rel} differs from stage_manifest.json"
STAGE = stage_manifest_path.parent / "stage"
CHAT_V1 = find_input("text150m_chat_v1.pt")
assert sha256(CHAT_V1) == CHAT_V1_SHA256, "text150m_chat_v1.pt is not round 1's chat model"


def config(name, out, **paths):
    cfg = json.loads((SRC / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    cfg.update({k: str(v) for k, v in paths.items()}, out_dir=str(out), name=out.name)
    path = Path(f"/tmp/{out.name}.json")
    path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return path


A, B = RUNS / "text-150m-explain-b3a", RUNS / "text-150m-explain-b3b"
TRIALS = [(config("text_150m_explain_v2", A, data_dir=STAGE / "ds7_v2_chat", init_from=CHAT_V1), A),
          (config("text_150m_explain_v2", B, data_dir=STAGE / "ds7_v2_weak_chat", init_from=CHAT_V1), B)]

In [ ]:
# One stage per GPU at the same time (each single-process), so both see the same session clock.
hours = SESSION_HOURS - (time.time() - T0) / 3600 - EVAL_HOURS - 0.2


def train(gpu, cfg, out):
    sh(sys.executable, "-m", "lm.train", "--config", str(cfg), "--max-hours", f"{hours:.2f}", log=f"{out.name}.log",
       env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % GPUS)))


threads = [threading.Thread(target=train, args=(gpu, cfg, out)) for gpu, (cfg, out) in enumerate(TRIALS)]
for t in threads:
    t.start()
    if GPUS == 1:
        t.join()
for t in threads:
    t.join()

In [ ]:
# The writer's dev tier (TRAINING_PLAN section 2; settings weakCases): the weak-case dev slice and devVal prompts of
# DS7 v2's validation split. Never the test sets: those are scored once per frozen candidate.
WEAK_SETTINGS = json.loads((SRC / "config" / "settings.json").read_text(encoding="utf-8"))["weakCases"]
DEV = [(STAGE / "dev" / "weak_dev.jsonl", "devweak", 100000), (STAGE / "dev" / "ds7v2_val.jsonl", "devval", WEAK_SETTINGS["devVal"])]


def dev_jobs(model, checkpoint):
    return [(model, checkpoint, test, tag, n, WEAK_SETTINGS["devValSeed"]) for test, tag, n in DEV]


def run_queue(gpu, queue):
    for model, checkpoint, test, tag, n, seed in queue:
        sh(sys.executable, "-m", "lm.eval_explain", "--model", model, "--checkpoint", str(checkpoint), "--test", str(test),
           "--tag", tag, "--n", str(n), "--seed", str(seed), log=f"eval-{model}-{tag}.log",
           env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))


def run_queues(queues):
    threads = [threading.Thread(target=run_queue, args=(gpu, q)) for gpu, q in enumerate(queues)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

jobs = dev_jobs("text150m_r1", base / "text150m_explain_v1.pt")
for name, out in (("text150m_b3a", A), ("text150m_b3b", B)):
    if (out / "model.pt").exists():
        jobs += dev_jobs(name, out / "model_best.pt")
# jobs alternate devweak / devval, so each dev file's jobs share one queue (they write one summary table)
run_queues([jobs[0::2], jobs[1::2]] if GPUS > 1 else [jobs])
tables = WORK / "reports" / "tables"
tables.mkdir(parents=True, exist_ok=True)
for f in (SRC / "reports" / "tables").glob("assistant_eval*_dev*.csv"):
    shutil.copy(f, tables)
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "gpus": GPUS, "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "numpy", "tokenizers")},
    "session_hours": round((time.time() - T0) / 3600, 2),
    "finished": {out.name: (out / "model.pt").exists() for _, out in TRIALS},
}, indent=2), encoding="utf-8")